# Task 1 — Denoising Autoencoder (DAE) on 100 grayscale 64×64 natural images

**What this notebook does (end to end):**
1. Builds a dataset of **100 training images** (grayscale, 64×64) from CIFAR-10, plus a disjoint **validation** set (for tuning) and a disjoint **test** set (touched only for the final evaluation).
2. Corrupts images with **Gaussian noise** (σ chosen so that content *and* noise are both clearly visible).
3. Trains a small **convolutional denoising autoencoder**, tunes it on the validation set, and evaluates it on train / validation / test with **MSE, PSNR and SSIM**.
4. Compares against two baselines (doing nothing, and a classical Gaussian blur filter).
5. Runs **correctness checks** (the task asks us to verify the implementation, not just report numbers).
6. Plots losses and shows qualitative examples, then discusses the results.

**How to run:** in Colab use *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Everything (data download included) is automatic. With a GPU the whole notebook takes a few minutes. To export the PDF: *File → Print → Save as PDF*.

## 0. Setup and configuration
All hyper-parameters that matter live in one `CFG` dictionary so it is clear what was chosen and what was tuned.

In [ ]:
import os, time, random, hashlib, copy, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
from scipy.ndimage import gaussian_filter

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42
def set_seed(seed=SEED):
    # Make runs reproducible (python, numpy, torch CPU + GPU)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU only")

CFG = dict(
    # ---- data ----
    n_train=100,     # the task: 100 training images
    n_val=500,       # held-out images used ONLY for tuning / model selection
    n_test=1000,     # held-out images used ONLY for the final evaluation
    img_size=64,     # CIFAR is 32x32 -> upsampled to 64x64 (see Section 2)
    # ---- noise ----
    sigma=0.2,       # std of the Gaussian noise, images live in [0, 1]
    # ---- training ----
    batch_size=10,
    epochs_tune=150,   # epochs per configuration in the hyper-parameter search
    epochs_final=600,  # epochs for the final model (best-validation checkpoint is kept)
    # ---- hyper-parameter grid (tuned on the validation set) ----
    grid=dict(
        base=[16, 32],      # channel width of the network
        latent=[8, 32],     # channels of the 8x8 bottleneck -> latent size = 64 * latent
        lr=[1e-3, 3e-4],    # Adam learning rate
        flip=[False, True], # random horizontal-flip augmentation
    ),
)

if device.type == "cpu":
    print("\n*** WARNING: no GPU found. Using a much smaller search so it finishes in reasonable time. ***")
    print("*** For the real run: Runtime > Change runtime type > T4 GPU, then run all again.        ***\n")
    CFG["epochs_tune"], CFG["epochs_final"] = 25, 80
    CFG["grid"] = dict(base=[16], latent=[8, 32], lr=[1e-3], flip=[True])

# Collects every correctness check so we can print one summary table at the end
CHECKS = []
def check(name, cond, detail=""):
    ok = bool(cond)
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    return ok

set_seed()

## 1. Task formulation

**Denoising autoencoder.** We have a clean image $x\in[0,1]^{64\times64}$. We build a corrupted version
$$\tilde x = x + n,\qquad n\sim\mathcal N(0,\sigma^2 I)$$
and train a network $f_\theta$ (encoder → low-dimensional bottleneck → decoder) so that $f_\theta(\tilde x)\approx x$. The loss is the mean squared error
$$\mathcal L(\theta)=\mathbb E_{x,n}\big[\|f_\theta(x+n)-x\|^2\big].$$

*Why this works / why a bottleneck:* an ordinary autoencoder with enough capacity can learn the identity. Because the **input is noisy but the target is clean**, the identity is no longer optimal, so the network must learn what natural images look like (smooth regions, edges) and keep only that. The bottleneck adds further pressure to keep only the important structure.

*Why MSE:* for additive Gaussian noise, minimising MSE is exactly maximum-likelihood. The MSE-optimal denoiser is the **posterior mean** $\mathbb E[x\mid\tilde x]$. This has a typical consequence we will see in the results: outputs are a bit *smooth/blurry* (when the model is unsure whether a fine detail is noise or texture, the average of the possibilities is smooth).

**Noise is re-sampled at every training step** (fresh $n$ each time an image is used). With only 100 images this is the key trick against over-fitting: the network sees a practically unlimited number of different noisy inputs, while the clean targets stay the same.

**Metrics** (all computed between the reconstruction and the *clean* image, per image, then averaged; pixel range $[0,1]$):
* **MSE** $=\frac1{HW}\sum (\hat x-x)^2$ — the training loss. Lower is better.
* **PSNR** $=10\log_{10}\!\big(1/\text{MSE}\big)$ dB — same information as MSE on a log scale; the standard image-denoising metric. Higher is better. Sanity anchor: noisy input with σ=0.2 has MSE≈σ²=0.04, i.e. PSNR≈13.98 dB.
* **SSIM** — structural similarity (local luminance, contrast and structure comparison, Gaussian window as in Wang et al. 2004), in [−1,1], 1 = identical. It correlates better with perceived quality than MSE.

**Data roles (no leakage):**
| split | size | role |
|---|---|---|
| train | 100 | the only images the network is trained on |
| validation | 500 | choose hyper-parameters and the best epoch |
| test | 1000 | final report only; never used for any decision |

## 2. Data: 100 grayscale natural images (CIFAR-10)

* CIFAR-10 contains natural photographs (animals, vehicles, ...). We ignore the labels.
* Colour → grayscale with the standard luma formula $0.299R+0.587G+0.114B$.
* CIFAR images are 32×32, the task asks for 64×64, so we **upsample with bicubic interpolation**. *Honest caveat:* upsampling adds pixels but no new fine detail, so these images are smoother than native 64×64 photos. White noise is high-frequency, so it is somewhat easier to separate from smooth images. This is discussed in the analysis.
* Train/val images are sampled from the CIFAR-10 *training* split (disjoint random indices), test images from the official CIFAR-10 *test* split. We additionally remove exact duplicate images across splits (by hashing) so there is no leakage.

In [ ]:
def to_gray_uint8(x_uint8):
    # (N,32,32,3) uint8 RGB -> (N,32,32) uint8 grayscale (ITU-R 601 luma)
    x = x_uint8.float()
    g = 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]
    return g.round().clamp(0, 255).to(torch.uint8)

def upsample(gray_uint8, size):
    # (N,32,32) uint8 -> (N,1,size,size) float32 in [0,1], bicubic interpolation
    x = gray_uint8.float().div(255).unsqueeze(1)
    x = F.interpolate(x, size=(size, size), mode="bicubic", align_corners=False)
    return x.clamp(0, 1)

def md5s(gray_uint8):
    return [hashlib.md5(a.numpy().tobytes()).hexdigest() for a in gray_uint8]

cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True)
cifar_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
g_train_all = to_gray_uint8(torch.from_numpy(cifar_train.data))
g_test_all  = to_gray_uint8(torch.from_numpy(cifar_test.data))
h_train_all, h_test_all = md5s(g_train_all), md5s(g_test_all)

rng = np.random.RandomState(SEED)

# --- training images: 100 uniformly random images from the CIFAR-10 train split
perm = rng.permutation(len(g_train_all))
idx_train = perm[:CFG["n_train"]]
seen = set(h_train_all[i] for i in idx_train)

# --- validation images: other images of the train split (skip exact duplicates of train images)
idx_val = []
for i in perm[CFG["n_train"]:]:
    if h_train_all[i] not in seen:
        idx_val.append(i); seen.add(h_train_all[i])
    if len(idx_val) == CFG["n_val"]: break

# --- test images: from the official test split (skip anything identical to a train/val image)
idx_test = []
for i in rng.permutation(len(g_test_all)):
    if h_test_all[i] not in seen:
        idx_test.append(i)
    if len(idx_test) == CFG["n_test"]: break

Xtr = upsample(g_train_all[idx_train], CFG["img_size"])
Xva = upsample(g_train_all[idx_val],   CFG["img_size"])
Xte = upsample(g_test_all[idx_test],   CFG["img_size"])

# Hard integrity checks: no image can be in two splits
H_tr = set(h_train_all[i] for i in idx_train)
H_va = set(h_train_all[i] for i in idx_val)
H_te = set(h_test_all[i]  for i in idx_test)
assert not (H_tr & H_va) and not (H_tr & H_te) and not (H_va & H_te), "data leakage between splits!"
print("train", tuple(Xtr.shape), "| val", tuple(Xva.shape), "| test", tuple(Xte.shape))

In [ ]:
def show_grid(x, n=10, title=None):
    fig, axes = plt.subplots(1, n, figsize=(1.4 * n, 1.6))
    for a, im in zip(axes, x[:n]):
        a.imshow(im[0].clamp(0, 1), cmap="gray", vmin=0, vmax=1); a.axis("off")
    if title: fig.suptitle(title, y=1.02)
    plt.show()

show_grid(Xtr, 10, "Some of the 100 clean training images (64x64 grayscale)")
show_grid(Xte, 10, "Some clean test images (never used for training)")

## 3. Noise model and choice of the noise level

Pixels are in $[0,1]$. We add i.i.d. Gaussian noise $n\sim\mathcal N(0,\sigma^2)$ and deliberately do **not** clip the noisy image: clipping would make the noise non-Gaussian at saturated pixels, and without clipping we get the clean theoretical identity MSE(noisy, clean)=σ² that we use as a correctness check later. (Images are only clipped to [0,1] for *display*.)

We look at several σ and pick the one where the **noise is clearly visible but the content is still clearly recognisable**, as required.

In [ ]:
def add_noise(x, sigma):
    # Fresh Gaussian noise (uses the global RNG of x's device; used during training)
    return x + sigma * torch.randn_like(x)

def make_fixed_noisy(x, sigma, seed):
    # Deterministic noisy copy of a CPU tensor (used for validation / test / reporting)
    g = torch.Generator().manual_seed(seed)
    return x + sigma * torch.randn(x.shape, generator=g)

sigmas = [0.05, 0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(3, len(sigmas) + 1, figsize=(1.7 * (len(sigmas) + 1), 5.3))
for r in range(3):
    axes[r, 0].imshow(Xte[r, 0], cmap="gray", vmin=0, vmax=1); axes[r, 0].axis("off")
    if r == 0: axes[r, 0].set_title("clean", fontsize=9)
    for c, s in enumerate(sigmas, start=1):
        noisy = make_fixed_noisy(Xte[r:r+1], s, seed=r)[0, 0].clamp(0, 1)
        axes[r, c].imshow(noisy, cmap="gray", vmin=0, vmax=1); axes[r, c].axis("off")
        if r == 0: axes[r, c].set_title(f"σ={s}", fontsize=9)
plt.suptitle("Choosing the noise level (display clipped to [0,1])", y=1.0)
plt.tight_layout(); plt.show()

**Choice: σ = 0.2** (≈ 51 grey levels out of 255). At σ ≤ 0.1 the noise is hardly noticeable; at σ ≥ 0.3 fine structures start to disappear; at σ = 0.2 the grain is obvious but objects/edges are still clearly visible. The *same* σ is used for train, validation and test.

Now the **fixed noisy versions** used for all evaluation (different seed per split, same σ). Training uses fresh noise at every step instead.

In [ ]:
SIGMA = CFG["sigma"]
Ntr_fix = make_fixed_noisy(Xtr, SIGMA, seed=1)   # train images + fixed noise, for *evaluating* on train
Nva_fix = make_fixed_noisy(Xva, SIGMA, seed=2)
Nte_fix = make_fixed_noisy(Xte, SIGMA, seed=3)

# keep tensors needed every epoch on the device
dev = lambda t: t.to(device)
Xtr_d, Xva_d, Xte_d = dev(Xtr), dev(Xva), dev(Xte)
Ntr_d, Nva_d, Nte_d = dev(Ntr_fix), dev(Nva_fix), dev(Nte_fix)

## 4. Metrics and baselines
* `mse_per_image`, `psnr_from_mse`, `ssim_per_image`: metrics described above (SSIM from scikit-image with the original Gaussian-window settings).
* **Baseline 1 – "do nothing":** use the noisy image as the output. Any useful denoiser must beat this.
* **Baseline 2 – Gaussian blur:** a classical denoiser (low-pass filter) whose width $\sigma_f$ is tuned on the *validation* set. This tells us whether the neural network learns something beyond simple smoothing.

In [ ]:
def mse_per_image(pred, target):
    return ((pred - target) ** 2).mean(dim=(1, 2, 3))

def psnr_from_mse(mse, peak=1.0):
    return 10 * torch.log10(peak ** 2 / mse)

def ssim_per_image(pred, target):
    p, t = pred.numpy(), target.numpy()
    return np.array([structural_similarity(p[i, 0], t[i, 0], data_range=1.0, gaussian_weights=True,
                                           sigma=1.5, use_sample_covariance=False) for i in range(len(p))])

def all_metrics(pred, target):
    # pred/target: CPU tensors (N,1,H,W). Returns per-image arrays.
    mse = mse_per_image(pred, target)
    return dict(mse=mse.numpy(), psnr=psnr_from_mse(mse).numpy(), ssim=ssim_per_image(pred, target))

@torch.no_grad()
def denoise(model, noisy, bs=250):
    # Run the model in eval mode on a (CPU or GPU) tensor, returns CPU tensor
    model.eval()
    return torch.cat([model(noisy[i:i + bs].to(device)).cpu() for i in range(0, len(noisy), bs)])

@torch.no_grad()
def mean_mse_dev(model, noisy_d, clean_d, bs=250):
    # Fast mean MSE for per-epoch tracking (data already on the device)
    model.eval()
    tot = 0.0
    for i in range(0, len(noisy_d), bs):
        tot += ((model(noisy_d[i:i + bs]) - clean_d[i:i + bs]) ** 2).mean(dim=(1, 2, 3)).sum().item()
    return tot / len(noisy_d)

def gaussian_filter_denoise(noisy, sigma_f):
    return torch.from_numpy(gaussian_filter(noisy.numpy(), sigma=(0, 0, sigma_f, sigma_f), mode="reflect")).float()

# Tune the blur width on the VALIDATION set only
filter_grid = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
filt_val = {s: mse_per_image(gaussian_filter_denoise(Nva_fix, s), Xva).mean().item() for s in filter_grid}
BEST_FILTER_SIGMA = min(filt_val, key=filt_val.get)
print("Gaussian-filter baseline, validation MSE per filter width:",
      {k: round(v, 5) for k, v in filt_val.items()})
print("-> best filter width:", BEST_FILTER_SIGMA)

## 5. Model: a small convolutional denoising autoencoder

```
input 1x64x64 (noisy)
 Encoder: [conv3x3 + ReLU, conv3x3 stride2 + ReLU]  x3   ->  channels c, 2c, 4c ; resolution 32, 16, 8
          1x1 conv (linear)                              ->  latent z x 8 x 8        (bottleneck, 64*z numbers)
 Decoder: [conv3x3 + ReLU, nearest-upsample x2]       x3   ->  resolution 16, 32, 64
          conv3x3 + ReLU, conv3x3, Sigmoid               ->  1x64x64 (output in [0,1])
```
* Strided convolutions downsample (learned pooling); the decoder uses *nearest-upsample + conv* instead of transposed convolutions, which avoids checkerboard artefacts.
* Sigmoid output keeps reconstructions in $[0,1]$ like the clean images.
* There are **no skip connections**: everything must pass through the bottleneck (adding them would turn it into a U-Net, which would denoise better but is no longer a classical autoencoder).
* The bottleneck size $64z$ vs. the 4096 input pixels gives a compression ratio of $64/z$ (z=8 → 8×, z=32 → 2×); $z$ and the width $c$ are tuned.

In [ ]:
class ConvDAE(nn.Module):
    def __init__(self, base=32, latent=16):
        super().__init__()
        c = base
        def enc(i, o):  # one encoder stage: keep resolution, then halve it
            return [nn.Conv2d(i, o, 3, padding=1), nn.ReLU(True), nn.Conv2d(o, o, 3, stride=2, padding=1), nn.ReLU(True)]
        def dec(i, o):  # one decoder stage: refine, then double the resolution
            return [nn.Conv2d(i, o, 3, padding=1), nn.ReLU(True), nn.Upsample(scale_factor=2, mode="nearest")]
        self.encoder = nn.Sequential(*enc(1, c), *enc(c, 2 * c), *enc(2 * c, 4 * c),
                                     nn.Conv2d(4 * c, latent, 1))                 # bottleneck: latent x 8 x 8
        self.decoder = nn.Sequential(*dec(latent, 4 * c), *dec(4 * c, 2 * c), *dec(2 * c, c),
                                     nn.Conv2d(c, c, 3, padding=1), nn.ReLU(True),
                                     nn.Conv2d(c, 1, 3, padding=1), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))

m = ConvDAE(base=32, latent=16).to(device)
with torch.no_grad():
    z, y = m.encoder(Xtr_d[:2]), m(Xtr_d[:2])
print("input", tuple(Xtr_d[:2].shape), "-> latent", tuple(z.shape), f"({z[0].numel()} numbers) -> output", tuple(y.shape))
print("parameters:", sum(p.numel() for p in m.parameters()))
del m, z, y

## 6. Training code
`train_model` does the following for one configuration:
* each epoch: shuffle the 100 training images, split into mini-batches, optionally flip horizontally, **add fresh Gaussian noise**, forward, MSE loss against the clean batch, Adam step;
* after every epoch evaluate (in eval mode, on the *fixed* noisy sets) the MSE of whatever sets are passed in `track`;
* remember the weights of the epoch with the lowest **validation** MSE (early-stopping-style checkpoint) and return them.

Only the 100 training images are ever used in the gradient step.

In [ ]:
def train_model(base, latent, lr, flip, epochs, track, seed=SEED, verbose=False):
    # track: dict name -> (noisy_fixed_on_device, clean_on_device). Must contain "val".
    set_seed(seed)
    model = ConvDAE(base, latent).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bs, n = CFG["batch_size"], len(Xtr_d)
    hist = {"epoch": [], "time": [], "train_running": [], **{k: [] for k in track}}
    best = dict(val=float("inf"), epoch=0, state=None)
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        run = 0.0
        for i in range(0, n, bs):
            xb = Xtr_d[perm[i:i + bs]]
            if flip:   # random horizontal flip (data augmentation of the CLEAN image)
                mask = (torch.rand(len(xb), device=device) < 0.5)[:, None, None, None]
                xb = torch.where(mask, xb.flip(-1), xb)
            loss = F.mse_loss(model(add_noise(xb, SIGMA)), xb)   # fresh noise at every step
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.item() * len(xb)
        hist["epoch"].append(ep); hist["train_running"].append(run / n)
        for k, (nz, cl) in track.items():
            hist[k].append(mean_mse_dev(model, nz, cl))
        if device.type == "cuda": torch.cuda.synchronize()
        hist["time"].append(time.time() - t0)
        if hist["val"][-1] < best["val"]:
            best.update(val=hist["val"][-1], epoch=ep, state={k: v.detach().clone() for k, v in model.state_dict().items()})
        if verbose and (ep % 25 == 0 or ep == 1):
            print(f"  epoch {ep:4d} | train(run) {hist['train_running'][-1]:.5f} | val {hist['val'][-1]:.5f} | {hist['time'][-1]:.0f}s")
    model.load_state_dict(best["state"])
    return model, hist, best

## 7. Correctness checks *before* training
The task asks us to verify that the implementation is correct. We do it in layers; every check prints PASS/FAIL and is collected in a summary table at the end.

**(a) The data and the noise are what we claim.**

In [ ]:
check("Data: all images are float32, 1x64x64, values in [0,1]",
      all(t.dtype == torch.float32 and t.shape[1:] == (1, 64, 64) and t.min() >= 0 and t.max() <= 1 for t in (Xtr, Xva, Xte)))
check("Data: train/val/test are pairwise disjoint (md5 of every image)", True,
      f"{len(H_tr)}/{len(H_va)}/{len(H_te)} unique images")
check("Data: exactly 100 training images", len(Xtr) == 100)

for name, clean, noisy in [("train", Xtr, Ntr_fix), ("val", Xva, Nva_fix), ("test", Xte, Nte_fix)]:
    r = noisy - clean
    check(f"Noise ({name}): mean≈0 and std≈σ={SIGMA}", abs(r.mean()) < 0.005 and abs(r.std() / SIGMA - 1) < 0.03,
          f"mean={r.mean():+.4f}, std={r.std():.4f}")
    mse_in = mse_per_image(noisy, clean).mean().item()
    check(f"Noisy-input MSE ({name}) ≈ σ² and PSNR ≈ {10*np.log10(1/SIGMA**2):.2f} dB", abs(mse_in / SIGMA**2 - 1) < 0.05,
          f"MSE={mse_in:.4f} vs σ²={SIGMA**2:.4f}, PSNR={psnr_from_mse(torch.tensor(mse_in)).item():.2f} dB")

**(b) The metric functions are correct** (compared with independent implementations and with known identities).

In [ ]:
a, b = Xte[:20], Nte_fix[:20]
check("MSE: own implementation == torch.nn.functional.mse_loss",
      abs(mse_per_image(b, a).mean().item() - F.mse_loss(b, a).item()) < 1e-6)
psnr_ref = np.mean([peak_signal_noise_ratio(a[i, 0].numpy(), b[i, 0].numpy(), data_range=1.0) for i in range(20)])
psnr_own = psnr_from_mse(mse_per_image(b, a)).mean().item()
check("PSNR: own implementation == scikit-image", abs(psnr_ref - psnr_own) < 1e-3, f"{psnr_own:.4f} vs {psnr_ref:.4f}")
check("SSIM: identical images give 1.0", np.allclose(ssim_per_image(a, a), 1.0, atol=1e-6))
check("SSIM: noisy image scores clearly lower than 1.0", ssim_per_image(b, a).mean() < 0.9, f"mean SSIM(noisy, clean)={ssim_per_image(b, a).mean():.3f}")

**(c) The model + training loop can learn** — the classic "overfit a tiny batch" test: with *no noise*, 8 images and enough steps, a correct network/optimiser/loss must drive the loss to ≈ 0. If this fails, there is a bug in the model, the loss, or the gradient flow, and any later result would be meaningless.

In [ ]:
set_seed(SEED)
tiny = ConvDAE(base=16, latent=16).to(device)
opt = torch.optim.Adam(tiny.parameters(), lr=1e-3)
xb = Xtr_d[:8]
losses = []
for step in range(500):
    loss = F.mse_loss(tiny(xb), xb)          # clean in -> clean out
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
check("Overfit-a-tiny-batch: loss falls by >50x to <1e-3", losses[-1] < 1e-3 and losses[-1] < losses[0] / 50,
      f"loss {losses[0]:.4f} -> {losses[-1]:.6f}")
plt.figure(figsize=(5, 2.8)); plt.semilogy(losses); plt.xlabel("step"); plt.ylabel("MSE")
plt.title("Sanity check: overfitting 8 clean images (no noise)"); plt.show()
del tiny, opt

## 8. Hyper-parameter tuning (validation set only)

**Procedure.** Grid search over network width, bottleneck size, learning rate and flip-augmentation. For each configuration we train for `epochs_tune` epochs on the 100 training images and record the **best validation MSE** (validation noise is fixed, so configurations are compared on exactly the same noisy images). The test set is *not* used here. The configuration with the lowest validation MSE is selected.

*Limitation:* each configuration is trained with a single seed, so very small differences between configurations are within run-to-run noise. The purpose is to choose a sensible configuration, not to claim the optimum.

In [ ]:
keys = list(CFG["grid"].keys())
combos = list(itertools.product(*[CFG["grid"][k] for k in keys]))
print(f"{len(combos)} configurations x {CFG['epochs_tune']} epochs\n")

tune_rows, tune_hist = [], {}
t_start = time.time()
for ci, vals in enumerate(combos, 1):
    hp = dict(zip(keys, vals))
    _, hist, best = train_model(**hp, epochs=CFG["epochs_tune"], track={"val": (Nva_d, Xva_d)})
    tune_rows.append({**hp, "best_val_mse": best["val"], "best_epoch": best["epoch"],
                      "final_train_running": hist["train_running"][-1], "seconds": hist["time"][-1]})
    tune_hist[vals] = hist
    print(f"[{ci:2d}/{len(combos)}] {hp} -> best val MSE {best['val']:.5f} (epoch {best['epoch']})")
print(f"\nTuning took {time.time() - t_start:.0f}s")

tune_df = pd.DataFrame(tune_rows).sort_values("best_val_mse").reset_index(drop=True)
BEST_HP = {k: tune_df.loc[0, k] for k in keys}
BEST_HP = {k: (bool(v) if k == "flip" else int(v) if k in ("base", "latent") else float(v)) for k, v in BEST_HP.items()}
display(tune_df.style.format({"best_val_mse": "{:.5f}", "final_train_running": "{:.5f}", "seconds": "{:.0f}"}))
print("Selected configuration:", BEST_HP)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
best_key = tuple(BEST_HP[k] for k in keys)
for vals, h in tune_hist.items():
    ax.plot(h["epoch"], h["val"], color="tab:red" if vals == best_key else "0.7",
            lw=2 if vals == best_key else 1, zorder=3 if vals == best_key else 1)
ax.axhline(SIGMA ** 2, color="k", ls="--", lw=1, label="noisy input (MSE = σ²)")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("validation MSE")
ax.set_title("Tuning: validation MSE of every configuration (red = selected)"); ax.legend(); plt.show()

## 9. Final training
We retrain the selected configuration for longer (`epochs_final`) on the **same 100 training images** and keep the weights from the epoch with the best **validation** MSE. 

We also log the **test** MSE after every epoch, *only to draw the requested train-vs-test curve*; it is never used to choose anything (model selection uses validation only).

Three training-related curves are drawn:
* *train (running)*: the actual training loss — measured on freshly noised (and possibly flipped) inputs while the weights are changing;
* *train (fixed noise)*: the same 100 images with a fixed noise draw, evaluated in eval mode — directly comparable with validation/test;
* *validation* and *test*: fixed noise, held-out images.

In [ ]:
print("Training final model with", BEST_HP)
model, hist, best = train_model(**BEST_HP, epochs=CFG["epochs_final"],
                                track={"train_fixed": (Ntr_d, Xtr_d), "val": (Nva_d, Xva_d), "test": (Nte_d, Xte_d)},
                                verbose=True)
print(f"\nBest validation epoch: {best['epoch']} (val MSE {best['val']:.5f}); total time {hist['time'][-1]:.0f}s")
print(f"MSE at the selected epoch -> train(fixed noise) {hist['train_fixed'][best['epoch']-1]:.5f} | "
      f"val {hist['val'][best['epoch']-1]:.5f} | test {hist['test'][best['epoch']-1]:.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, xkey, xlabel in zip(axes, ["epoch", "time"], ["epoch", "training time (s, wall-clock)"]):
    ax.plot(hist[xkey], hist["train_running"], color="tab:blue", alpha=0.35, label="train (running, fresh noise)")
    ax.plot(hist[xkey], hist["train_fixed"], color="tab:blue", label="train (fixed noise)")
    ax.plot(hist[xkey], hist["val"], color="tab:green", label="validation")
    ax.plot(hist[xkey], hist["test"], color="tab:red", label="test")
    ax.axhline(SIGMA ** 2, color="k", ls="--", lw=1, label="noisy input (σ²)")
    if xkey == "epoch": ax.axvline(best["epoch"], color="gray", ls=":", label="selected epoch (best val)")
    ax.set_yscale("log"); ax.set_xlabel(xlabel); ax.set_ylabel("MSE loss"); ax.legend(fontsize=8)
axes[0].set_title("Loss vs epoch"); axes[1].set_title("Loss vs training time")
plt.tight_layout(); plt.show()

## 10. Final evaluation on train and test (table)
For every split we compute MSE, PSNR and SSIM (mean ± standard deviation across images) for: noisy input, the Gaussian-filter baseline and the DAE. All splits use the **same noise level σ**; the test set was never used for training or selection.

In [ ]:
splits = {"Train (100)": (Xtr, Ntr_fix), "Validation (500)": (Xva, Nva_fix), "Test (1000)": (Xte, Nte_fix)}
recon, M = {}, {}
for name, (clean, noisy) in splits.items():
    recon[name] = {"Noisy input (no denoising)": noisy,
                   f"Gaussian filter (σ_f={BEST_FILTER_SIGMA})": gaussian_filter_denoise(noisy, BEST_FILTER_SIGMA),
                   "Denoising autoencoder": denoise(model, noisy)}
    M[name] = {meth: all_metrics(pred, clean) for meth, pred in recon[name].items()}

rows = []
for name in splits:
    for meth, m_ in M[name].items():
        rows.append({"Split": name, "Method": meth,
                     "MSE": f"{m_['mse'].mean():.5f} ± {m_['mse'].std():.5f}",
                     "PSNR (dB)": f"{m_['psnr'].mean():.2f} ± {m_['psnr'].std():.2f}",
                     "SSIM": f"{m_['ssim'].mean():.3f} ± {m_['ssim'].std():.3f}"})
results_df = pd.DataFrame(rows).set_index(["Split", "Method"])
display(results_df)

In [ ]:
# Compact view of the DAE's improvement per split
summ = pd.DataFrame({name: {"DAE MSE": M[name]["Denoising autoencoder"]["mse"].mean(),
                            "Noisy-input MSE": M[name]["Noisy input (no denoising)"]["mse"].mean(),
                            "PSNR gain over noisy (dB)": M[name]["Denoising autoencoder"]["psnr"].mean() - M[name]["Noisy input (no denoising)"]["psnr"].mean(),
                            "SSIM gain over noisy": M[name]["Denoising autoencoder"]["ssim"].mean() - M[name]["Noisy input (no denoising)"]["ssim"].mean()}
                     for name in splits}).T
display(summ.style.format({"DAE MSE": "{:.5f}", "Noisy-input MSE": "{:.5f}", "PSNR gain over noisy (dB)": "{:+.2f}", "SSIM gain over noisy": "{:+.3f}"}))

## 11. Qualitative results
Rows: clean image, noisy input, DAE reconstruction (with PSNR in the title), and the absolute error ×4 (so remaining errors are visible). Test images were **not** used in training. The first set of examples is simply the *first* images of each set (no cherry-picking); afterwards we also show the **worst** test images, to see failure cases.

In [ ]:
def show_examples(clean, noisy, pred, idxs, title):
    n = len(idxs)
    fig, axes = plt.subplots(4, n, figsize=(1.55 * n, 6.6))
    psnr = psnr_from_mse(mse_per_image(pred[idxs], clean[idxs])).numpy()
    rows = [("clean", clean), ("noisy input", noisy.clamp(0, 1)), ("DAE output", pred), ("|error| x4", (pred - clean).abs() * 4)]
    for r, (lab, data) in enumerate(rows):
        for c, i in enumerate(idxs):
            axes[r, c].imshow(data[i, 0].clamp(0, 1), cmap="gray", vmin=0, vmax=1); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            axes[r, c].grid(False)
            if r == 2: axes[r, c].set_title(f"{psnr[c]:.1f} dB", fontsize=8)
        axes[r, 0].set_ylabel(lab, fontsize=9)
    fig.suptitle(title, y=0.995); plt.tight_layout(); plt.show()

D = "Denoising autoencoder"
show_examples(Xtr, Ntr_fix, recon["Train (100)"][D], list(range(8)), "TRAIN images (used for training)")
show_examples(Xte, Nte_fix, recon["Test (1000)"][D], list(range(8)), "TEST images (never seen in training) - first 8, not cherry-picked")
worst = np.argsort(M["Test (1000)"][D]["psnr"])[:8].tolist()
show_examples(Xte, Nte_fix, recon["Test (1000)"][D], worst, "TEST images - the 8 WORST by PSNR (failure cases)")

In [ ]:
# Distribution over images (not just the mean): how much does the DAE improve each test image?
gain = M["Test (1000)"][D]["psnr"] - M["Test (1000)"]["Noisy input (no denoising)"]["psnr"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(gain, bins=40, color="tab:red", alpha=0.8); axes[0].axvline(0, color="k")
axes[0].set_xlabel("PSNR gain of DAE over noisy input (dB), per test image"); axes[0].set_ylabel("# images")
axes[0].set_title(f"Improved {np.mean(gain > 0) * 100:.1f}% of test images")
for name, c in [("Train (100)", "tab:blue"), ("Test (1000)", "tab:red")]:
    axes[1].hist(M[name][D]["psnr"], bins=30, alpha=0.55, color=c, density=True, label=f"{name}, mean {M[name][D]['psnr'].mean():.2f} dB")
axes[1].set_xlabel("PSNR of DAE output (dB)"); axes[1].set_ylabel("density"); axes[1].legend(); axes[1].set_title("PSNR distribution: train vs test")
plt.tight_layout(); plt.show()

## 12. Correctness checks *after* training
These checks answer: *is the trained network really denoising the specific input image, and are the numbers trustworthy?*

1. **Beats doing nothing / beats the mean image.** A model that just outputs a constant average image would also have a low-ish MSE, so we compare with that explicitly.
2. **Output depends on the input (negative control).** If we compare the DAE output for image *i* to the clean image of a *different* image *j*, the error must be much larger than to its own clean image. If it were not, the network would be ignoring its input.
3. **Not a lucky noise draw.** Re-evaluate the test set with several different noise seeds; the result must barely move.
4. **Output range** is within [0,1].
5. **Noise-level mismatch** (Section 13) behaves as theory predicts.

In [ ]:
Mte = M["Test (1000)"]
mse_dae, mse_noisy = Mte[D]["mse"].mean(), Mte["Noisy input (no denoising)"]["mse"].mean()
mse_filt = Mte[f"Gaussian filter (σ_f={BEST_FILTER_SIGMA})"]["mse"].mean()
check("Test: DAE MSE < noisy-input MSE", mse_dae < mse_noisy, f"{mse_dae:.5f} < {mse_noisy:.5f}")

mean_img = Xtr.mean(0, keepdim=True).expand_as(Xte)       # constant prediction = mean of the TRAINING images
mse_mean_img = mse_per_image(mean_img, Xte).mean().item()
check("Test: DAE MSE < 'predict the mean training image' MSE", mse_dae < mse_mean_img, f"{mse_dae:.5f} < {mse_mean_img:.5f}")

check("Test: DAE MSE < tuned Gaussian-filter MSE (performance check, not a bug check)", mse_dae < mse_filt, f"{mse_dae:.5f} vs {mse_filt:.5f}")

pred_te = recon["Test (1000)"][D]
g = torch.Generator().manual_seed(0)
shuffled = torch.randperm(len(Xte), generator=g)
mse_wrong = mse_per_image(pred_te, Xte[shuffled]).mean().item()
check("Negative control: DAE output vs clean image of ANOTHER image is >3x worse", mse_wrong > 3 * mse_dae,
      f"matched {mse_dae:.5f} vs mismatched {mse_wrong:.5f}")

seed_mses = [mse_per_image(denoise(model, make_fixed_noisy(Xte, SIGMA, seed=100 + s)), Xte).mean().item() for s in range(5)]
check("Test MSE is stable across 5 different noise draws (std < 2% of mean)", np.std(seed_mses) < 0.02 * np.mean(seed_mses),
      f"mean {np.mean(seed_mses):.5f}, std {np.std(seed_mses):.6f}")
check("DAE outputs lie in [0,1]", pred_te.min() >= 0 and pred_te.max() <= 1)

print(f"\nReference MSEs on the test set:  noisy {mse_noisy:.5f} | mean image {mse_mean_img:.5f} | "
      f"Gaussian filter {mse_filt:.5f} | DAE {mse_dae:.5f}")

## 13. Robustness to the noise level (extra analysis)
The network was trained at σ = 0.2 only. How does it behave when the test noise level differs? (Same test images; the Gaussian filter keeps the width tuned at σ = 0.2 for a fair "fixed method" comparison.) A denoiser trained for one noise level is expected to over-smooth at lower noise and under-denoise at higher noise.

In [ ]:
sweep = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]
res = {"noisy": [], "filter": [], "dae": []}
for s in sweep:
    nz = make_fixed_noisy(Xte, s, seed=7)
    res["noisy"].append(psnr_from_mse(mse_per_image(nz, Xte).mean()).item())
    res["filter"].append(psnr_from_mse(mse_per_image(gaussian_filter_denoise(nz, BEST_FILTER_SIGMA), Xte).mean()).item())
    res["dae"].append(psnr_from_mse(mse_per_image(denoise(model, nz), Xte).mean()).item())
plt.figure(figsize=(6.5, 4))
plt.plot(sweep, res["noisy"], "k--o", label="noisy input"); plt.plot(sweep, res["filter"], "g-s", label=f"Gaussian filter (σ_f={BEST_FILTER_SIGMA}, fixed)")
plt.plot(sweep, res["dae"], "r-o", label="DAE (trained at σ=0.2)")
plt.axvline(SIGMA, color="gray", ls=":"); plt.xlabel("test noise level σ"); plt.ylabel("PSNR (dB) on the test set"); plt.legend()
plt.title("Behaviour under noise-level mismatch"); plt.show()

## 14. Analysis and discussion
The numbers below are generated from the run above, so the text always matches the results.

In [ ]:
tr, te = M["Train (100)"], M["Test (1000)"]
tr_mse, te_mse = tr[D]["mse"].mean(), te[D]["mse"].mean()
gap = te_mse / tr_mse
db_gain_te = te[D]["psnr"].mean() - te["Noisy input (no denoising)"]["psnr"].mean()
db_gain_tr = tr[D]["psnr"].mean() - tr["Noisy input (no denoising)"]["psnr"].mean()
filt_key = f"Gaussian filter (σ_f={BEST_FILTER_SIGMA})"
db_vs_filter = te[D]["psnr"].mean() - te[filt_key]["psnr"].mean()

print("PERFORMANCE")
print(f"  * Noisy input has MSE {te['Noisy input (no denoising)']['mse'].mean():.4f} (= σ² = {SIGMA**2:.4f}), PSNR {te['Noisy input (no denoising)']['psnr'].mean():.2f} dB, SSIM {te['Noisy input (no denoising)']['ssim'].mean():.3f}.")
print(f"  * The DAE reaches test MSE {te_mse:.5f}, PSNR {te[D]['psnr'].mean():.2f} dB, SSIM {te[D]['ssim'].mean():.3f}: "
      f"a gain of {db_gain_te:+.2f} dB PSNR and {te_mse / te['Noisy input (no denoising)']['mse'].mean() * 100:.0f}% of the noisy MSE remains.")
print(f"  * It improved {np.mean(gain > 0) * 100:.1f}% of the individual test images.")
if db_vs_filter > 0:
    print(f"  * It beats the best classical Gaussian filter by {db_vs_filter:.2f} dB on the test set, so the network learned more than simple smoothing.")
else:
    print(f"  * It does NOT beat the best classical Gaussian filter on the test set ({db_vs_filter:+.2f} dB): with only 100 training images, "
          f"and smooth (upsampled) images, a plain low-pass filter is a very strong baseline.")

print("\nGENERALISATION")
print(f"  * Train MSE {tr_mse:.5f} vs test MSE {te_mse:.5f} -> test/train ratio {gap:.2f}; PSNR gain on train {db_gain_tr:+.2f} dB vs test {db_gain_te:+.2f} dB.")
if gap < 1.15:
    print("  * The gap is small: the network did not memorise its 100 training images. Fresh noise at every step, augmentation, "
          "and picking the epoch by validation MSE all limit over-fitting.")
elif gap < 1.6:
    print("  * There is a moderate generalisation gap: the network does fit the training images somewhat better than unseen ones, "
          "which is expected with only 100 training images, but it still transfers well to unseen images.")
else:
    print("  * There is a large generalisation gap: the network fits the 100 training images much better than unseen images (over-fitting).")
print(f"  * The validation-selected epoch is {best['epoch']} of {CFG['epochs_final']}"
      + (" (i.e. validation loss stopped improving before the end, early-stopping helped)." if best['epoch'] < CFG['epochs_final'] - 5 else " (still improving at the end, more epochs might help)."))

**How to interpret these results (reasoning that holds for this kind of experiment):**

* **What the model learned.** The network never saw the noise-free *test* images as targets, yet it removes most of the noise on them. It therefore learned a general prior for "what natural images look like" (smooth areas separated by edges) rather than memorising 100 pictures. The qualitative figures support this: the structure survives and the grain disappears.
* **Why outputs look smooth.** With the MSE loss the best answer for an uncertain pixel is the posterior *mean*. Fine textures that are indistinguishable from noise at σ=0.2 are averaged out, so reconstructions are somewhat blurrier than the clean images. The error images (|error|×4) show that the remaining error concentrates on edges and textured regions; the worst-case test images are the highly textured/high-contrast ones, where noise and detail overlap.
* **Generalisation vs. over-fitting.** With just 100 training images, a network with several hundred thousand parameters *can* over-fit. The generalisation gap between the training and test rows of the table (and between the curves) measures this. The measures that keep it in check are: new noise at every step (the noise acts as unlimited data augmentation for the inputs), horizontal flips (if selected by the tuning), a bottleneck, and choosing the epoch by validation MSE.
* **Noise-level mismatch.** The robustness plot shows what the network really learned: a denoiser tuned to σ=0.2. At much lower noise it removes too much (over-smoothing) and at much higher noise it leaves residual noise. That is a limitation of training at a single fixed σ (a *blind* denoiser would be trained on a range of σ).
* **Caveat about the data.** CIFAR images were upsampled from 32×32 to 64×64, so they have less fine detail than native 64×64 photos. That makes the noise easier to separate from the signal and makes the Gaussian-filter baseline relatively stronger than it would be on native high-resolution images.
* **Limitations / extensions.** Single seed per configuration in the search; PSNR/SSIM are only proxies for visual quality; a U-Net (skip connections), training on a σ range, or a perceptual loss would all improve the result and are natural next steps.

## 15. How the implementation was verified (summary)
All checks that were run in this notebook, in one table:

In [ ]:
checks_df = pd.DataFrame(CHECKS)
display(checks_df.style.apply(lambda col: ["background-color: #d4edda" if v == "PASS" else "background-color: #f8d7da" if v == "FAIL" else "" for v in col], subset=["result"]))
print(f"{(checks_df.result == 'PASS').sum()} / {len(checks_df)} checks passed")

**Verification logic in words**

1. **Inputs are right:** shapes/ranges/dtypes, 100 training images, train/val/test pairwise disjoint (hash-checked), and the noise really is Gaussian with mean 0 and std σ at the *same* level for every split. The exact identity MSE(noisy, clean)=σ² (PSNR≈13.98 dB at σ=0.2) holds, which confirms both the noise generator and the metric code.
2. **Metrics are right:** my MSE/PSNR equal PyTorch/scikit-image; SSIM(x,x)=1.
3. **The learning machinery works:** the model can overfit a tiny noise-free batch to ≈0 loss (gradients, loss and optimiser are wired correctly).
4. **No leakage and honest evaluation:** only the 100 training images enter the gradient step; hyper-parameters and the best epoch are chosen on the validation set; the test set is used only for reporting (its per-epoch curve is logged for plotting but never used for decisions).
5. **The trained model really denoises *the given input*:** better than noisy input, than a constant mean image and (reported honestly) than a tuned Gaussian filter; and the output for image *i* matches clean image *i* far better than clean image *j≠i* (negative control); the result is stable across noise draws.